In [2]:
"""Dataset utilities for the brain-encoding baseline experiments.

The dataset stores preprocessed stimulus videos and the corresponding neural
and behavioral recordings as NumPy arrays, with one file per sample.
"""

import os
import numpy as np
import torch
from torch.utils.data import Dataset


class MouseDataset(Dataset):
    """Map preprocessed NumPy samples to PyTorch tensors.

    The dataset expects four parallel directories under ``root_dir``:
    ``videos``, ``pupil_center``, ``behavior``, and ``labels``. File names are
    numeric identifiers and must match across all directories.

    Args:
        root_dir: Root directory containing the four modality folders.
    """

    def __init__(self, root_dir):
        """Initialize the dataset and index the available sample identifiers.

        Args:
            root_dir: Root directory containing the ``videos``,
                ``pupil_center``, ``behavior``, and ``labels`` subdirectories.
        """
        self.videos_dir = os.path.join(root_dir, 'videos')
        self.pupil_dir = os.path.join(root_dir, 'pupil_center')
        self.behavior_dir = os.path.join(root_dir, 'behavior')
        self.labels_dir = os.path.join(root_dir, 'labels')

        self.file_ids = sorted(
            [f.replace('.npy', '') for f in os.listdir(self.videos_dir) if f.endswith('.npy')],
            key=lambda x: int(x)
        )

    def __len__(self):
        """Return the number of samples in the dataset."""
        return len(self.file_ids)

    def __getitem__(self, i):
        """Load one sample and return its modality tensors.

        Args:
            i: Integer index of the sample.

        Returns:
            A dictionary containing the ``video``, ``pupil_center``,
            ``behavior``, and ``labels`` tensors. The video receives an
            additional channel dimension, and the one-dimensional modalities
            are transposed to match the model input layout.
        """
        file_id = self.file_ids[i]

        video = np.load(os.path.join(self.videos_dir, f'{file_id}.npy'))
        pupil = np.load(os.path.join(self.pupil_dir, f'{file_id}.npy'))
        behavior = np.load(os.path.join(self.behavior_dir, f'{file_id}.npy'))
        label = np.load(os.path.join(self.labels_dir, f'{file_id}.npy'))

        video = video[np.newaxis, ...]
        
        return {
            'video': torch.from_numpy(video).float(),
            'pupil_center': torch.from_numpy(pupil).float().transpose(1,0),
            'behavior': torch.from_numpy(behavior).float().transpose(1,0),
            'labels': torch.from_numpy(label).float()
        }

In [3]:
# Dataset roots exposed by the Kaggle input mount.
train_set = MouseDataset('/kaggle/input/mousedatasetct/train/data')
validation_set = MouseDataset('/kaggle/input/mousedatasetct/validation/data')
test_set = MouseDataset('/kaggle/input/mousedatasetct/test/data')

In [4]:
train_set[0]['labels'].shape

torch.Size([227, 140])

# Baseline Evaluation

**Training set layout**

* Samples 0-41 belong to *natural\_movie\_one*: one clip every 6 samples.
* Samples 42-216 belong to *natural\_movie\_three*: one clip every 25 samples.


In [5]:
import numpy as np
import statistics

def retrieve_dff_mean_among_trials_for_a_clip(movie_type: str , train_set: MouseDataset, clip: int):
    """Compute the mean neural response of one clip across training trials.

    Args:
        movie_type: Movie identifier. Use ``"one"`` for
            ``natural_movie_one`` or ``"three"`` for
            ``natural_movie_three``.
        train_set: Training dataset that provides the ``labels`` tensors.
        clip: One-based clip index. Valid values are 1-6 for
            ``natural_movie_one`` and 1-25 for ``natural_movie_three``.

    Returns:
        A two-dimensional array of shape ``(227, 140)`` containing the
        trial-averaged response for the requested clip.

    Raises:
        AssertionError: If ``clip`` is outside the valid range for the
            selected movie.
        ValueError: If ``movie_type`` is neither ``"one"`` nor ``"three"``.
    """
    
    offset = 6 if movie_type=="one" else 25

    # Validate the requested clip against the selected movie.
    if movie_type == "one":
        # natural_movie_one occupies sample indices 0-41.
        assert 1 <= clip <= 6, f"Per movie_type='one', clip deve essere tra 1 e 6, ricevuto: {clip}"
    elif movie_type == "three":
        # natural_movie_three occupies sample indices 42-216.
        assert 1 <= clip <= 25, f"Per movie_type='three', clip deve essere tra 1 e 25, ricevuto: {clip}"
    else:
        raise ValueError(f"movie_type deve essere 'one' o 'three', ricevuto: {movie_type}")
    
    trials = []
    start_idx = clip-1 if movie_type == "one" else clip+41
    for i in range(7):
        trials.append(train_set[start_idx]['labels'].numpy())
        start_idx += offset
    
    # Average over trials: (7, 227, 140) -> (227, 140).
    mean_across_trials = np.mean(trials, axis=0)
    return mean_across_trials

In [6]:
# 62 samples: 12 for natural_movie_one (trial 8) and 50 for
# natural_movie_three (trial 8).
print(len(test_set))


62


In [7]:
# Retrieve test trials 8 and 9 for clip 1 of natural_movie_one.
# The baseline is compared against each trial independently.

clip_1_type_one_test_trial8 = test_set[0]['labels'] 
clip_1_type_one_test_trial9 = test_set[6]['labels'] 

In [8]:
# Retrieve test trials 8 and 9 for clip 1 of natural_movie_three.
# The baseline is compared against each trial independently.

clip_1_type_three_test_trial8 = test_set[13]['labels'] 
clip_1_type_three_test_trial9 = test_set[38]['labels'] 

In [9]:
# Pearson correlation between the trial-averaged training response for
# clip 1 and the corresponding test trials.

import numpy as np

def compute_neuron_correlations(mean_training, test_trial):
    """Compute the per-neuron Pearson correlation between two responses.

    Neurons with near-zero variance in either input are skipped because their
    correlation coefficient is undefined.

    Args:
        mean_training: Trial-averaged training response of shape
            ``(num_neurons, num_frames)``.
        test_trial: Test response of shape ``(num_neurons, num_frames)``.

    Returns:
        A dictionary containing the mean correlation over valid neurons, the
        number of valid neurons, and the total number of neurons.
    """
    correlations = []

    print(mean_training.shape)
    for neuron in range(mean_training.shape[0]):
        # Extract the time series of the current neuron.
        train_neuron = mean_training[neuron, :]  # shape (140,)
        test_neuron = test_trial[neuron, :]      # shape (140,)
        
        # Skip neurons whose activity is effectively constant.
        if np.var(train_neuron) > 1e-10 and np.var(test_neuron) > 1e-10:
            corr = np.corrcoef(train_neuron, test_neuron)[0, 1]
            if not np.isnan(corr) and np.isfinite(corr):
                correlations.append(corr)
    
    return {
        'mean_correlation': np.mean(correlations) if correlations else 0.0,
        'valid_neurons': len(correlations),
        'total_neurons': mean_training.shape[0]
    }

In [ ]:
def pearson_correlation(predictions, labels):
    """Compute the mean single-trial correlation over neurons.

    The last 66 frames of both tensors are used for temporal alignment.
    Neurons with near-zero variance in either the prediction or the label are
    excluded so that the Pearson coefficient remains well defined.

    Args:
        predictions: Predicted responses of shape
            ``(num_neurons, num_frames)``.
        labels: Ground-truth responses of shape
            ``(num_neurons, num_frames)``.

    Returns:
        A dictionary containing the ``eval_single_trial_correlation`` metric.
    """

    # Align predictions and labels to the last 66 frames.
    min_frames = 66
    labels_aligned = labels[..., -min_frames:]
    predictions_aligned = predictions[..., -min_frames:]
    
    y_true = labels_aligned.transpose(1, 0) # (time, neurons)
    y_pred = predictions_aligned.transpose(1,0) # (time, neurons)
        
    correlations = []
        
    for neuron in range(y_true.shape[1]): 
        # Compare the predicted and recorded time series of each neuron.
        true_vals = y_true[:, neuron]
        pred_vals = y_pred[:, neuron]
        
        true_var = np.var(true_vals)
        pred_var = np.var(pred_vals)
        
        # Neurons with near-zero variance are filtered out before computing
        # the correlation. This avoids division by zero in the Pearson
        # coefficient: inactive neurons, which are common in neural datasets,
        # have undefined correlations and are therefore excluded from the
        # metric. Only neurons with sufficient variability in both labels and
        # predictions contribute to the average, which keeps the metric
        # statistically meaningful.

        if true_var > 1e-10 and pred_var > 1e-10:
            corr = np.corrcoef(true_vals, pred_vals)[0, 1]
            if not np.isnan(corr) and np.isfinite(corr):
                correlations.append(corr)
    
    mean_correlation = np.mean(correlations) if correlations else 0.0
        
    return {
        'eval_single_trial_correlation': mean_correlation,
    }

## Sample-Based Approach


The baseline is computed on the training split, which contains seven trials (indices 0-6) for both **natural\_movie\_one** and **natural\_movie\_three**. The segmentation step yields 6 clips for **natural\_movie\_one** and 25 clips for **natural\_movie\_three**; each clip has shape (227, 140).


In [11]:
# Pearson correlation between the clip 1 training mean and test trial 8.
clip_1_type_one_train = retrieve_dff_mean_among_trials_for_a_clip("one", train_set, 1)
result = pearson_correlation(clip_1_type_one_train, clip_1_type_one_test_trial8.numpy())
print(result) 

{'eval_single_trial_correlation': 0.1503593076233057}


In [12]:
# Pearson correlation between the clip 1 training mean and test trial 9.
result = pearson_correlation(clip_1_type_one_train, clip_1_type_one_test_trial9.numpy())
print(result) 

{'eval_single_trial_correlation': 0.15572641561251682}


In [13]:
# Pearson correlation between the clip 1 training mean and test trial 8.
clip_1_type_three_train = retrieve_dff_mean_among_trials_for_a_clip("three", train_set, 1)
result = pearson_correlation(clip_1_type_three_train, clip_1_type_three_test_trial8.numpy())
print(result) # 0.01626231821295263

{'eval_single_trial_correlation': 0.014238362417651177}


In [14]:
# Pearson correlation between the clip 1 training mean and test trial 9.
result = pearson_correlation(clip_1_type_three_train, clip_1_type_three_test_trial9.numpy())
print(result) # 0.025573463105490166

{'eval_single_trial_correlation': 0.017345757218436755}


The test trials for the same clip are then retrieved (trials 8 and 9). The Pearson correlation is computed between the NumPy array obtained above, which contains the trial-averaged intensity of each neuron for the selected clip, and the corresponding clip from trial 8 and trial 9, respectively.


## Average of Single-Trial Correlations


Computing the baseline over all clips in the dataset is the more accurate approach. It provides more representative information than the sample-based approach and matches the model-selection metric: the `compute_metrics` method called by the Hugging Face Trainer returns the average single-trial correlation across all samples in the test\_set. The value produced by the model's `evaluate()` on the test\_set is therefore directly comparable to this baseline. The correlation is first computed between the train\_set responses and the test\_set clips from trial 8, then between the train\_set responses and the test\_set clips from trial 9.


In [15]:
# Accumulate the trial-averaged response of all 31 clips.
list_array_numpy = []
for movie_type in ["one","three"]:
    _range = 6 if movie_type == "one" else 25
    for i in range(_range):
        list_array_numpy.append(retrieve_dff_mean_among_trials_for_a_clip(movie_type, train_set, i+1))

print(list_array_numpy[0].shape)
# Stack the per-clip arrays into a single 3D array.
array_3d_train = np.stack(list_array_numpy, axis=0)
print(array_3d_train.shape)  # (31, 227, 140)

(227, 140)
(31, 227, 140)


In [16]:
# Split the test set into trial 8 and trial 9.
# The first 31 samples contain 6 clips from natural_movie_one and 25 clips
# from natural_movie_three for trial 8; the remaining samples belong to
# trial 9.
trial_8_list = []
trial_9_list = []
for i in range(len(test_set)):
    if i < 31:
        trial_8_list.append(test_set[i]['labels'].numpy())
    else:
        trial_9_list.append(test_set[i]['labels'].numpy())

array_3d_trial8_test = np.stack(trial_8_list, axis=0)
array_3d_trial9_test = np.stack(trial_9_list, axis=0)
print(array_3d_trial8_test.shape)  # (31, 227, 140)
print(array_3d_trial9_test.shape)  # (31, 227, 140)

(31, 227, 140)
(31, 227, 140)


In [ ]:
def compute_metrics(predictions, labels):
    """Compute the average single-trial correlation over a batch.

    Args:
        predictions: Predicted responses of shape
            ``(batch_size, num_neurons, num_frames)``.
        labels: Ground-truth responses of shape
            ``(batch_size, num_neurons, num_frames)``.

    Returns:
        A dictionary containing the mean correlation, its standard deviation
        across examples, and the number of evaluated examples.
    """
    min_frames = 66
    labels_aligned = labels[..., -min_frames:]
    predictions_aligned = predictions[..., -min_frames:]
    
    # Store the mean correlation of each example.
    example_correlations = [] 
    
   
    for example_idx in range(labels_aligned.shape[0]):  # First dimension: batch size.
       
        y_true_example = labels_aligned[example_idx].T  # (time, neurons)
        y_pred_example = predictions_aligned[example_idx].T  # (time, neurons)
        
        # Correlations of the valid neurons in this example.
        correlations = [] 
        

        for neuron in range(y_true_example.shape[1]):
            true_vals = y_true_example[:, neuron]
            pred_vals = y_pred_example[:, neuron]
            
           
            true_var = np.var(true_vals)
            pred_var = np.var(pred_vals)
            
            if true_var > 1e-10 and pred_var > 1e-10:
                corr = np.corrcoef(true_vals, pred_vals)[0, 1]
                if not np.isnan(corr) and np.isfinite(corr):
                    correlations.append(corr)
        
  
        mean_corr_example = np.mean(correlations) if correlations else 0.0
        example_correlations.append(mean_corr_example) 
    
   
    # Aggregate the per-example correlations.
    overall_mean = np.mean(example_correlations) if example_correlations else 0.0
    
    return {
        'eval_average_single_trial_correlation': overall_mean,
        'eval_single_trial_std': np.std(example_correlations) if example_correlations else 0.0,
        'eval_num_examples': len(example_correlations),
    }

In [18]:
import statistics

# Evaluate the baseline against both test trials.
results = [
    compute_metrics(array_3d_train, array_3d_trial8_test)["eval_average_single_trial_correlation"],
    compute_metrics(array_3d_train, array_3d_trial9_test)["eval_average_single_trial_correlation"]
]
print(results)

[0.007475139404624604, 0.039593989755850795]


In [19]:
print(f"Baseline among all data: {statistics.mean(results)} ")

Baseline among all data: 0.0235345645802377 
